In [17]:
import os
import json
import pandas as pd
import numpy as np
import glob

# Define the root directory
root_dir = 'results_2511'

# List to store the extracted data
all_records = []

# Walk through the directory structure
# We assume structure: results_2511/strategy/language/model_folder/json_file

invalid_names = []
for root, dirs, files in os.walk(root_dir):
    # print("Processing directory:", root)
    for file in files:
        if file.endswith('.json'):

            
            file_path = os.path.join(root, file)
            
            if "zero_shot+specific" in root:
                continue  # Skip specific model runs if needed
            
            # Read JSON content
            with open(file_path, 'r') as f:
                content = json.load(f)
            
            # Extract sections
            metadata = content.get('metadata', {})
            data_list = content.get('data', [])
            
            # Skip if data list is empty
            if not data_list:
                continue
                
            # --- 1. Extract Info from Folder Structure (as requested) ---
            # Get path parts to identify language and model string
            path_parts = os.path.normpath(file_path).split(os.sep)
            
            # Assuming depth: .../language/model_folder/file.json
            # Python negative indexing is safer:
            language_from_folder = path_parts[-3] if len(path_parts) >= 3 else "unknown"
            model_folder_name = path_parts[-2] if len(path_parts) >= 2 else "unknown"
            
            # Parse "code_llm" and "answer_llm" from folder name
            # Format: "generator+verifier__v1"
            if '+' in model_folder_name:
                parts = model_folder_name.split('__')[0].split('+')
                folder_code_llm = parts[0]
                folder_answer_llm = parts[1] if len(parts) > 1 else "unknown"
            else:
                folder_code_llm = model_folder_name
                folder_answer_llm = "unknown"

            # --- 2. Process Ground Truth (GT) ---
            # Logic: NaN -> Yes, "No" -> No
            raw_gt = metadata.get('answer')
            
            gt_normalized = "Unknown"
            
            # Handle Float NaN, None, or String "NaN"
            is_nan = False
            if raw_gt is None:
                is_nan = True
            elif isinstance(raw_gt, float) and np.isnan(raw_gt):
                is_nan = True
            elif str(raw_gt).strip() == "NaN":
                is_nan = True
                
            if is_nan:
                gt_normalized = "Yes"
            elif str(raw_gt).lower().strip() == "no":
                gt_normalized = "No"
            else:
                # Keep original if it's something unexpected (e.g. "Yes")
                gt_normalized = str(raw_gt)

            # --- 3. Process Model Prediction ---
            # Taking the first iteration from 'data' list
            model_data = data_list[0]
            prediction_code = model_data.get('answer')
            # if prediction_code != "C1": continue

            # assert False, "Incomplete code snippet provided."

            msg = model_data.get('exec_msg', '')

            invalid_name = get_collection_name(msg)
            if invalid_name is not None:
                invalid_names.append(invalid_name)

            # if "not found" in script.lower():
            #     assert False, "Incomplete code snippet provided."

print(invalid_names)

['LANDSAT/LC08/C02/SR', 'MODIS/061/FIRMS', 'NOAA/VIIRS/001/VNP14IMGTDL_NRT', 'COPERNICUS/CAMS/EAC4/MONTHLY', 'LANDSAT/LC08/C02/SR', 'NASA/TRMM/3B43V7', 'NOAA/PERSIANN_CDR', 'MODIS/051/MCD19A2_GRANULAR_AOD', 'VIIRS/001/VNP14A1', 'JRC/GHSL/P2023A/GHS_SMOD_POP', 'NOAA/CMDL/OZONE_SH_V1', 'NOAA/VIIRS/DNB/V2/VCMSLFG', 'NOAA/VIIRS/DNB/DAILY_V1/VCMSLCFG', 'NOAA/VIIRS/DNB/VNP46A2', 'NOAA/CDR/OISST/V2.1', 'NOAA/VIIRS/DNB/VNP46A2', 'NOAA/CDR/OISST/V2.1', 'LANDSAT/LT05/C02/T1_SR', 'NOAA/VIIRS/001/VNP14IMGML', 'NOAA/CDR/OISST/V2.1', 'NOAA/CDR/OISST/V2.1', 'NASA/AURA/OMI/OMTO3e', 'JAXA/AHI/L1B/V20190124', 'MODIS/061/MCD14ML', 'NOAA/VIIRS/DNB/DAILY', 'MODIS/061/MCD14DL', 'MODIS/061/MCD14DL', 'ESA/CCI/MonthlyLandSurfaceSoilMoisture', 'MODIS/006/MYD04_L2', 'MODIS/006/MYD28M', 'NASA/NSIDC_MEASURES/V001/MOD10CM', 'MODIS/006/MYD04_L2', 'COPERNICUS/S5P/NRTI_L3_SO2', 'UCSB-CHG/CHIRPS/Monthly', 'MODIS/006/MCD14DL', 'JPL/OCEAN/MUR/G1SST', 'MODIS/061/MCD14DL', 'UCSB-CHG/CHIRPS/Monthly', 'MODIS/061/MOD04_3K', '

In [ ]:
hallucinated and correct
LANDSAT/LC08/C02/SR, LANDSAT/LC08/C02/T1_L2
NOAA/VIIRS/001/VNP14IMGTDL_NRT, NASA/LANCE/SNPP_VIIRS/C2
NASA/TRMM/3B43V7, TRMM/3B43V7
MODIS/051/MCD19A2_GRANULAR_AOD, MODIS/061/MCD19A2_GRANULES"
JRC/GHSL/P2023A/GHS_SMOD_POP, JRC/GHSL/P2023A/GHS_SMOD_POP'

COPERNICUS/S5P/NRTI_L3_SO2, COPERNICUS/S5P/NRTI/L3_SO2
UCSB-CHG/CHIRPS/Monthly, UCSB-CHG/CHIRPS/PENTAD

SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (2262603713.py, line 3)

In [16]:
# use regex to extract collection name
# for example the message can be "ImageCollection asset 'NOAA/VIIRS/001/VNP14IMGTDL_NRT' not found", we want to extract "NOAA/VIIRS/001/VNP14IMGTDL_NRT"


def get_collection_name(msg):

    import re
    patterns = [r"ImageCollection asset '([^']+)' not found", r"Collection asset '([^']+)' not found"]
    match = None
    for pattern in patterns:
        match = re.search(pattern, msg)
        if match:
            break
    if match:
        collection_name = match.group(1)
        return collection_name
    return None

msg = "Collection.loadTable: Collection asset 'MODIS/006/MCD14DL' not found.\n    at module$contents$ee$apiclient_apiclient.handleRespons"
get_collection_name(msg)

'MODIS/006/MCD14DL'

In [4]:
print(model_data.keys())
print(model_data["exec_msg"])
print(model_data["code"])

dict_keys(['iter', 'raw_code', 'code', 'exec_msg', 'exec_stderr', 'exec_returncode', 'raw_answer', 'answer', 'answer_thinking'])
Historical November Fire Counts (2012-2023):
  Error retrieving data for 2012: ImageCollection.load: ImageCollection asset 'NOAA/VIIRS/001/VNP14IMGTDL_NRT' not found (does not exist or caller does not have access).
  Error retrieving data for 2013: ImageCollection.load: ImageCollection asset 'NOAA/VIIRS/001/VNP14IMGTDL_NRT' not found (does not exist or caller does not have access).
  Error retrieving data for 2014: ImageCollection.load: ImageCollection asset 'NOAA/VIIRS/001/VNP14IMGTDL_NRT' not found (does not exist or caller does not have access).
  Error retrieving data for 2015: ImageCollection.load: ImageCollection asset 'NOAA/VIIRS/001/VNP14IMGTDL_NRT' not found (does not exist or caller does not have access).
  Error retrieving data for 2016: ImageCollection.load: ImageCollection asset 'NOAA/VIIRS/001/VNP14IMGTDL_NRT' not found (does not exist or caller

In [ ]:
question = content["data"][0]["code"]

In [36]:
# read /Users/ck696/Documents/GitHub/univearth_acl/results_2511/zero_shot/python/claude-sonnet-4-5-20250929+gemini-2.5-flash__v1/memory_32.json"""

import json
with open("/Users/ck696/Documents/GitHub/univearth_acl/results_2511/zero_shot/python/claude-sonnet-4-5-20250929+gemini-2.5-flash__v1/memory_384.json", 'r') as f:
    content = json.load(f)

data = content["data"][0]

In [37]:
print(data["code"])

import ee
ee.Initialize(project='earthsense-436113')

# 1. Geometry & Time Definitions
# Mount Lewotobi Laki-Laki location
volcano_point = ee.Geometry.Point([122.775, -8.533])
# Buffer to analyze plume area (50km radius)
analysis_region = volcano_point.buffer(50000)

# July 7, 2025, around 2 PM local time (UTC+8, so ~06:00 UTC)
# Use a window around that time
date_start = '2025-07-07'
date_end = '2025-07-08'

print(f"Analyzing volcanic plume for {date_start} near Mount Lewotobi Laki-Laki")

# 2. Data Acquisition - Try multiple sources
# Try Sentinel-2 first (10-30m resolution, 5-day revisit)
s2_col = ee.ImageCollection('COPERNICUS/S2_HARMONIZED') \
    .filterBounds(volcano_point) \
    .filterDate(date_start, date_end)

s2_count = s2_col.size().getInfo()
print(f"Sentinel-2 images found: {s2_count}")

# Try Landsat 8/9 as backup
l8_col = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2') \
    .filterBounds(volcano_point) \
    .filterDate(date_start, date_end)

l9_col = ee.ImageCollection('

In [35]:
print(data["exec_msg"])

August 2019 images: 31
August baseline images (1985-2012): 868
2019 SST: 1723.2945970405415, Pixels: 28359
Baseline SST: 1558.997852152977, Pixels: 28359
SST values outside realistic range
<answer>C3</answer>

